[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab1_Basic_RAG_LlamaParse_Solutions.ipynb)


# Lab 1 SOLUTIONS: Basic RAG with LlamaParse
## CCI Session 6

**Duration:** 15 minutes

### Clinical scenario
> KHCC pediatric oncologists need quick answers from long Wilms tumor guidelines. You parse the PDF, chunk and embed the text, store vectors, and answer with a grounded RAG chain.

### Objectives
- Parse a complex PDF with **LlamaParse** (layout-aware markdown)
- **Chunk** and **embed** for retrieval
- Store in **Chroma** and run a **retrieval QA** chain
- Inspect retrieved chunks (transparency / “citations”)


---
## Setup

Install packages, add **`OPENAI_API_KEY`** and **`LLAMA_CLOUD_API_KEY`** to Colab **Secrets**, then upload **`WT.pdf`**. If imports fail after install, use **Runtime → Restart session**.


In [ ]:
!pip install -q llama-parse llama-index langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb pypdf

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')

In [ ]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = '/content/WT.pdf'

---
## Cell 1 — Parse with LlamaParse

**LlamaParse** (LlamaCloud) converts the PDF to **markdown** with layout awareness — tables, lists, and headings stay usable for chunking and retrieval.

**Alternatives:** naive `PyPDFLoader` (Cell 2), **Docling**, **Azure Document Intelligence**, or open layout parsers. Trade-offs: cloud cost + data residency vs. local CPU quality.


In [ ]:
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type='markdown',
    verbose=True,
)
documents = parser.load_data(PDF_PATH)

print(f'Number of documents: {len(documents)}')
print(documents[0].text[:1500])

In [ ]:
import json
from google.colab import files

# Save parsed markdown to disk so later labs can skip re-parsing
parsed_export = [{'text': d.text, 'metadata': d.metadata} for d in documents]
with open('wt_parsed.json', 'w', encoding='utf-8') as f:
    json.dump(parsed_export, f, ensure_ascii=False)

files.download('wt_parsed.json')
print(f"wt_parsed.json downloaded ({len(parsed_export)} docs) — keep it with WT.pdf and chroma_wt.zip")

## Cell 2 — Naive PyPDF Comparison

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
naive_docs = PyPDFLoader(PDF_PATH).load()
print(f'Pages (PyPDF): {len(naive_docs)}')
print('--- PyPDF (jumbled) ---')
print(naive_docs[5].page_content[:1500])
print('\n--- LlamaParse (markdown) ---')
print(documents[0].text[1500:3000])

In [ ]:
# Cell 2b — Side-by-side parser comparison
# Renders PyPDF (naive) vs LlamaParse (layout-aware) in two HTML columns
# so you can visually judge which one preserves tables, headings, and lists.

from IPython.display import HTML, display
import html

# Pick a region that's likely to contain a table or structured content.
# Adjust these if you want to inspect a different section of the guideline.
PYPDF_PAGE   = 14      # which PyPDF page to show
LLAMA_START  = 0   # char offset into the LlamaParse markdown
LLAMA_LEN    = 4500   # how many chars to show

pypdf_text  = naive_docs[PYPDF_PAGE].page_content[1200:]
llama_text  = documents[14].text[LLAMA_START:LLAMA_START + LLAMA_LEN]

def _panel(title, subtitle, text, bg):
    return f"""
    <div style="flex:1; min-width:0; background:{bg}; border:1px solid #d0d7de;
                border-radius:8px; padding:12px; margin:4px;">
      <div style="font-weight:600; font-size:14px; margin-bottom:2px;">{title}</div>
      <div style="font-size:11px; color:#57606a; margin-bottom:8px;">{subtitle}</div>
      <pre style="white-space:pre-wrap; word-wrap:break-word; font-size:11px;
                  line-height:1.4; max-height:600px; overflow:auto; margin:0;
                  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;">{html.escape(text)}</pre>
    </div>
    """

display(HTML(f"""
<div style="display:flex; flex-wrap:wrap; gap:4px;">
  {_panel("PyPDFLoader (naive)",
          f"page {PYPDF_PAGE} • {len(pypdf_text)} chars • no layout awareness",
          pypdf_text, "#fff5f5")}
  {_panel("LlamaParse (markdown)",
          f"chars {LLAMA_START}–{LLAMA_START+LLAMA_LEN} • layout-aware",
          llama_text, "#f0f7ff")}
</div>
<div style="font-size:11px; color:#57606a; margin-top:8px;">
  Look for: table structure, heading hierarchy, list bullets, column order,
  and whether sentences are broken across the page.
</div>
"""))

---
## Cell 3 — Chunking for RAG

**Why chunk?** Embeddings and vector search work on *segments* of text, not whole PDFs. Chunks should be big enough to hold a complete fact (e.g. a dosing table row + context) but small enough to fit several in the LLM context.

**Common strategies (LangChain):**
- **`RecursiveCharacterTextSplitter`** (used here): splits on a priority list (`\n\n`, `\n`, `. `, ` `) so paragraphs and sentences stay intact when possible. Good default for clinical prose and markdown from LlamaParse.
- **`CharacterTextSplitter`**: fixed-size windows; fast but can cut mid-sentence — worse for guidelines.
- **Semantic / heading-based splitters**: split on section titles or embeddings; fewer cuts inside facts, more setup work.

**Tuning `chunk_size` and `chunk_overlap`:**
- Larger chunks (e.g. 1500–2000): more surrounding context per hit; fewer total chunks; risk of diluting relevance.
- Smaller chunks (e.g. 400–600): sharper retrieval; more risk of missing cross-sentence dependencies.
- **Overlap** (often 10–20% of chunk size): reduces the chance that a critical sentence sits on a chunk boundary.

This notebook uses **1000 / 200** as a balanced default for guideline text.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in documents]

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(lc_docs)

print(f'Chunks: {len(chunks)}')
print(chunks[10].page_content[:600])

---
## Cell 4 — Embedding models

Embeddings turn each chunk into a **vector** so “similar meaning” is “similar direction” in space.

**OpenAI options (common in teaching / prod):**
- **`text-embedding-3-small`**: good quality, lower cost and dimension (often 1536). **Default choice** for labs.
- **`text-embedding-3-large`**: stronger retrieval on hard queries; higher cost and vector size.
- **`text-embedding-ada-002`**: older; still usable but v3 models are preferred for new work.

**Alternatives:** Cohere, Voyage, open models (e.g. sentence-transformers) for on-prem or offline; pick one and keep the **same** model for index + query.


In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

---
## Cell 5 — Vector store (Chroma) + export

**Chroma** stores vectors on disk under `persist_directory`. After building the index, **zip and download `chroma_wt.zip`** — upload it in Labs 2 and 3 so you never have to re-parse or re-embed the same PDF again.

**Other stacks:** FAISS (in-memory, fast), or hosted vector DBs (Pinecone, Weaviate) for production.

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='wt_good',
    persist_directory='./chroma_wt',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print(f"Vector store saved → ./chroma_wt  (collection='wt_good', {vectorstore._collection.count()} chunks)")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('chroma_wt', 'zip', '.', 'chroma_wt')
files.download('chroma_wt.zip')
print("chroma_wt.zip downloaded — keep it alongside WT.pdf to reuse in Labs 2 and 3")

In [ ]:
# ── Download WT_parsed.md to your PC ─────────────────────────────────────────────────────────────
from google.colab import files

combined_md = '\n\n---\n\n'.join(d.text for d in documents)
with open('WT_parsed.md', 'w', encoding='utf-8') as f:
    f.write(combined_md)
files.download('WT_parsed.md')
print(f"WT_parsed.md downloaded ({len(combined_md):,} chars)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────

# ── Save JSON + vector store to Google Drive (Labs 2–3 load from here) ───────────────────────────
import os, shutil
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CCI_Session6'
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy('wt_parsed.json', f'{DRIVE_DIR}/wt_parsed.json')
shutil.copy('chroma_wt.zip',  f'{DRIVE_DIR}/chroma_wt.zip')
print(f"Also saved to Google Drive → {DRIVE_DIR}")
print(f"  wt_parsed.json — LangChain Document list (Labs 2–3)")
print(f"  chroma_wt.zip  — vector store (Labs 2–3)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────

## Cell 6 — RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a clinical assistant. Answer the question using ONLY the provided context. If unknown, say so.\n\nContext:\n{context}'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def _rag(d):
    docs = retriever.invoke(d['input'])
    answer = (prompt | llm | StrOutputParser()).invoke({'context': format_docs(docs), 'input': d['input']})
    return {'answer': answer, 'context': docs}

rag_chain = RunnableLambda(_rag)

for q in [
    'What is the standard treatment for Stage III favorable histology Wilms tumor?',
    'What are the most common side effects of vincristine?'
]:
    res = rag_chain.invoke({'input': q})
    print(f'\nQ: {q}\nA: {res["answer"]}')

## Cell 7 — Citations

In [ ]:
q1 = 'What is the standard treatment for Stage III favorable histology Wilms tumor?'
res = rag_chain.invoke({'input': q1})
for i, doc in enumerate(res['context']):
    print(f'--- Chunk {i+1} ---')
    print(doc.page_content[:300])
    print()

## Stretch — different chunk sizes

In [ ]:
for cs in [500, 2000]:
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=int(cs*0.2))
    ch = sp.split_documents(lc_docs)
    print(f'chunk_size={cs} → {len(ch)} chunks')